# Paired comparisons: benchmark, tuning and ablation


Wilcoxon p-values follow SciPy 1.11.4 automatic selection. Zero differences use its normal approximation, with the Wilcox zero convention; Holm correction is applied within each comparison family.


In [1]:
from pathlib import Path
import sys, os, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/run_model.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "experiments")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import final_visualization as fv
import result_tables as th
fv.apply_style()
from statistical_methods import holm, paired_scope
from Wilcoxon_test.wilcoxon_test import paired_wilcoxon as wilcoxon


## Dataset blocks, Wilcoxon, Holm and Friedman


In [2]:
sources = th.load_aligned_sources()
main = sources["benchmark_with_biological"]
complete = sources["complete"]
collected = [[] for _ in range(5)]

def add(frame, experiment, scope, metric, higher=True):
    for target, table in zip(collected, paired_scope(frame, experiment, scope, metric, higher)):
        table["experiment"] = experiment
        table["scope"] = scope
        table["metric_name"] = metric
        target.append(table)

for metric, higher in [("f1", True), ("SHD", False), ("fbeta", True)]:
    for family, frame in main.groupby("dataset", observed=True):
        add(frame, "benchmark", family, metric, higher)
    for arm in ("tuned", "untuned"):
        selected = complete.loc[complete.arm.eq(arm)]
        for family, frame in selected.groupby("dataset", observed=True):
            add(frame, arm, family, metric, higher)
        for scope, families in [
            ("Test", th.TEST_FAMILIES),
            ("OOD", th.OOD_FAMILIES),
            ("Test + OOD", th.SIX_FAMILIES),
        ]:
            add(
                selected.loc[selected.dataset.isin(families)],
                arm,
                scope,
                metric,
                higher,
            )
names = ["paired_results", "friedman", "rankings", "coverage", "dataset_blocks"]
comparison_tables = {name: pd.concat(pieces, ignore_index=True) for name, pieces in zip(names, collected)}
# Paired tuned vs untuned comparisons: Holm across models per family.
gains = []
for metric in ("f1", "fbeta", "SHD"):
    for family, group in complete.groupby("dataset", observed=True):
        for model, frame in group.groupby("model", observed=True):
            if not frame.finished.all():
                continue
            cells = frame.groupby(["dataset_seed", "arm"])[metric].mean().unstack("arm")
            if not {"tuned", "untuned"} <= set(cells):
                continue
            cells = cells.dropna(subset=["tuned", "untuned"])
            delta = cells.tuned - cells.untuned
            r = None if (delta == 0).all() else wilcoxon(delta)
            gains.append(
                dict(
                    dataset=family,
                    model=model,
                    metric=metric,
                    n_pairs=len(delta),
                    mean_tuned=float(cells.tuned.mean()),
                    mean_untuned=float(cells.untuned.mean()),
                    mean_difference=float(delta.mean()),
                    statistic=0.0 if r is None else float(r.statistic),
                    pvalue=1.0 if r is None else float(r.pvalue),
                )
            )


In [3]:
gains = pd.DataFrame(gains)
gains["pvalue_holm"] = gains.groupby(["dataset", "metric"]).pvalue.transform(lambda v: holm(v))
gains["significant_holm"] = gains.pvalue_holm < 0.05
favorable = np.where(
    gains.metric.eq("SHD"), gains.mean_difference < 0, gains.mean_difference > 0
)
gains["winner"] = np.where(
    gains.significant_holm,
    np.where(favorable, "tuned", "untuned"),
    "no_significant_difference",
)
gain_family = th.TEST_FAMILIES[0]
display(gains.loc[gains.dataset.eq(gain_family) & gains.metric.eq("fbeta")])
# Numeric ablation exports retain the final complete-fit exclusion policy.
from condition_data import load_results
fits = load_results("ablation").assign(family=lambda frame: frame.dataset)
fits = fits.loc[fits.status.eq("finished") & fits.fbeta.notna()]
cells = fits.groupby(["condition_id", "family", "dataset_seed"]).fbeta.mean().reset_index()
macro = (
    cells.groupby(["condition_id", "dataset_seed"])
    .filter(lambda g: g.family.nunique() == 3)
    .groupby(["dataset_seed", "condition_id"])
    .fbeta.mean()
    .unstack("condition_id")
)
comparisons = []
for condition in macro.columns:
    if condition == "full":
        continue
    pair = macro[["full", condition]].dropna()
    delta = pair["full"] - pair[condition]
    r = None if (delta == 0).all() else wilcoxon(delta)
    comparisons.append(
        dict(
            condition=condition,
            n_pairs=len(pair),
            mean_full=float(pair["full"].mean()),
            mean_condition=float(pair[condition].mean()),
            loss=float(delta.mean()),
            statistic=0.0 if r is None else float(r.statistic),
            pvalue=1.0 if r is None else float(r.pvalue),
        )
    )
table = pd.DataFrame(comparisons)
table["pvalue_holm"] = holm(table.pvalue)
table["winner"] = np.where(
    table.pvalue_holm < 0.05,
    np.where(table.loss > 0, "full", table.condition),
    "no_significant_difference",
)


,dataset,model,metric,n_pairs,mean_tuned,mean_untuned,mean_difference,statistic,pvalue,pvalue_holm,significant_holm,winner
126,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,AVICI,fbeta,15,0.604367,0.561643,0.042724,11.0,0.003357,0.033569,True,tuned
127,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,DAGMA_Linear,fbeta,15,0.376222,0.201103,0.175119,0.0,0.000061,0.000854,True,tuned
128,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,DAGMA_Nonlinear,fbeta,15,0.416315,0.390118,0.026197,50.0,0.599487,1.000000,False,no_significant_difference
129,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,DAS,fbeta,15,0.303392,0.288474,0.014919,33.0,0.135376,0.676880,False,no_significant_difference
130,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,DirectLiNGAM,fbeta,15,0.397014,0.322790,0.074224,11.0,0.003357,0.033569,True,tuned
131,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,FCI,fbeta,15,0.378754,0.297126,0.081629,11.0,0.003357,0.033569,True,tuned
132,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,GES,fbeta,15,0.536402,0.494095,0.042307,13.0,0.005371,0.037598,True,tuned
133,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,GOLEM,fbeta,15,0.460368,0.173799,0.286569,0.0,0.000061,0.000854,True,tuned
134,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,GRaSP,fbeta,15,0.656936,0.656936,0.000000,0.0,1.000000,1.000000,False,no_significant_difference
135,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,GraNDAG,fbeta,15,0.292385,0.109632,0.182753,14.0,0.006714,0.040283,True,tuned


In [4]:
display(table)


,condition,n_pairs,mean_full,mean_condition,loss,statistic,pvalue,pvalue_holm,winner
0,glasso_no_direction,15,0.614231,0.416630,0.197601,0.0,0.000061,0.000793,full
1,glasso_only,15,0.614231,0.603676,0.010555,23.0,0.035339,0.070679,no_significant_difference
2,hessian_no_direction,15,0.614231,0.415119,0.199112,0.0,0.000061,0.000793,full
3,hessian_only,15,0.614231,0.614409,-0.000178,51.0,0.638672,0.638672,no_significant_difference
4,matched_constant,15,0.614231,0.035333,0.578897,0.0,0.000061,0.000793,full
5,no_direction,15,0.614231,0.417563,0.196668,0.0,0.000061,0.000793,full
6,no_existence,14,0.611234,0.222552,0.388683,0.0,0.000122,0.000793,full
7,no_prior,15,0.614231,0.160579,0.453651,0.0,0.000061,0.000793,full
8,oracle,15,0.614231,0.966673,-0.352442,0.0,0.000061,0.000793,oracle
9,permuted_direction,15,0.614231,0.366669,0.247561,0.0,0.000061,0.000793,full


## Numerical comparisons and ranks

All comparisons are calculated above from individual fits. Model seeds are averaged within dataset instances; Holm adjustments are separate by experiment, scope and metric. Select a context below to inspect its ranks and adjusted pairwise p-values.


In [5]:
comparison_tables.update(tuning_comparisons=gains, ablation_comparisons=table)
summary = (comparison_tables["paired_results"].groupby(["experiment", "metric"])
           .agg(contexts=("scope", "nunique"), comparisons=("n_pairs", "size"),
                significant_comparisons=("significant_holm", "sum")).reset_index())
display(summary)

# Choose the experiment, dataset family/scope and metric to inspect.
experiment = "benchmark"
scope = th.OOD_FAMILIES[0]
metric = "f1"

def selected(frame):
    return frame.loc[frame.experiment.eq(experiment) & frame.scope.eq(scope)
                     & frame.metric.eq(metric)].copy()

ranks = selected(comparison_tables["rankings"]).sort_values(["rank", "model"])
display(ranks[["model", "mean", "wins", "losses", "n_blocks", "rank"]])
display(selected(comparison_tables["friedman"])[["n_blocks", "n_models", "statistic", "pvalue"]])
pairs = selected(comparison_tables["paired_results"])
pvalues = pairs.pivot(index="model_a", columns="model_b", values="pvalue_holm")
pvalues = pvalues.combine_first(pvalues.T).reindex(index=ranks.model, columns=ranks.model)
np.fill_diagonal(pvalues.values, 1.0)
display(pvalues.style.format("{:.3g}", na_rep="—"))


,experiment,metric,contexts,comparisons,significant_comparisons
0,benchmark,SHD,10,1036,696
1,benchmark,f1,10,1036,603
2,benchmark,fbeta,10,1036,492
3,tuned,SHD,9,945,445
4,tuned,f1,9,945,500
5,tuned,fbeta,9,945,455
6,untuned,SHD,9,945,541
7,untuned,f1,9,945,591
8,untuned,fbeta,9,945,480


,model,mean,wins,losses,n_blocks,rank
8,GRaSP,0.794479,14,0,15,1
0,AVICI,0.446184,11,1,15,2
4,DirectLiNGAM,0.333148,11,1,15,2
6,GES,0.393808,11,1,15,2
1,DAGMA_Linear,0.189695,7,4,15,5
10,NOTEARS,0.194446,6,4,15,5
12,PC,0.167993,2,4,15,5
13,Random,0.185594,3,4,15,5
2,DAGMA_Nonlinear,0.127557,1,5,15,9
3,DAS,0.107358,0,7,15,9


,n_blocks,n_models,statistic,pvalue
0,15,15,160.333333,6.081550e-27


model,GRaSP,AVICI,DirectLiNGAM,GES,DAGMA_Linear,NOTEARS,PC,Random,DAGMA_Nonlinear,DAS,FCI,GOLEM,NoGAM,SCORE,GraNDAG
model,,,,,,,,,,,,,,,
GRaSP,1,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641
AVICI,0.00641,1,0.161,1,0.0165,0.00641,0.00641,0.00641,0.0103,0.00641,0.00641,0.00641,0.00641,0.00757,0.00641
DirectLiNGAM,0.00641,0.161,1,1,0.0275,0.00641,0.0165,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641,0.00641
GES,0.00641,1,1,1,0.0367,0.0275,0.00641,0.00757,0.00757,0.00641,0.00641,0.00757,0.00757,0.00641,0.00641
DAGMA_Linear,0.00641,0.0165,0.0275,0.0367,1,1,1,1,0.0205,0.00641,0.0475,0.0475,0.00757,0.00641,0.0103
NOTEARS,0.00641,0.00641,0.00641,0.0275,1,1,1,1,0.058,0.00641,0.00641,0.0165,0.0205,0.0165,0.00641
PC,0.00641,0.00641,0.0165,0.00641,1,1,1,1,1,0.874,0.0165,1,1,1,0.00641
Random,0.00641,0.00641,0.00641,0.00757,1,1,1,1,0.137,0.0475,0.058,0.0165,0.0725,0.111,0.00641
DAGMA_Nonlinear,0.00641,0.0103,0.00641,0.00757,0.0205,0.058,1,0.137,1,1,1,1,1,1,0.0367
